# 모두마켓 분기 성과 요약 보고서

## 1. 분석 개요
▸ 대상: 모두마켓 주문 1,500건 (2025-01 ~ 2025-05), 고객·상품 정보 병합  
▸ 방법: 3개 테이블 키 검증 후 m:1 병합 → 지역별, 멤버십별, 카테고리별 KPI 집계 → 교차표·점유율 분석

## 2. 핵심 발견
▸ 통합 매출이 가장 높았던 지역은 인천, 멤버십 기준으로는 Basic에서 가장 많은 매출 발생    
▸ Basic, Premium 등급에서는 인천의 비중이 가장 크지만 Vip등급에서는 경기의 비중이 가장 큼  
▸ 카테고리 중 패션의 매출 기여가 가장 컸음, 객단가도 74,840으로 카테고리 중 가장 높음

## 3. 데이터 신뢰성 (반드시 기록)
▸ 병합 전 키 검증: 중복 0건, 미매칭 0건 → 행 폭증·조용한 결측 없음  
▸ validate="m:1" 통과 → 관계 가정(주문:상품 = m:1) 유효  

## 4. 제언
▸ 지역별 매출비중이 가장 큰 Order 상위 10건을 추출하여 각 order의 구매 항목과 멤버십을 비교, 추후 고객 세그멘테이션에 활용.

In [24]:
# 지역 x 회원등급 매출 교차 피벗

region_membership = pd.pivot_table(
    df, index="region", columns="membership", values="amount",
    aggfunc="sum", fill_value=0, margins=True, margins_name="합계",
).round(0)
print("[지역 × 회원등급 매출 교차표]")
display(region_membership)

# 카테고리별 KPI(매출·건수·객단가) 요약표

category_kpi = (
    df.groupby("category")
    .agg(
        총매출=("amount", "sum"),
        주문건수=("order_id", "count"),
        객단가=("amount", "mean"),        
    )
    .round(0)
)
print("[카테고리별 KPI]")
display(category_kpi)

# 각 주문이 '그 지역 전체 매출'에서 차지하는 비중 (transform)

df["region_total"] = df.groupby("region")["amount"].transform("sum")
df["share_in_region(%)"] = (df["amount"] / df["region_total"] * 100).round(3)
print("\n[주문별 지역 매출 점유율 — 앞 10행]")
display(df[["order_id", "region", "amount", "share_in_region(%)"]].head(10))

[지역 × 회원등급 매출 교차표]


membership,basic,premium,vip,합계
region,,,,
경기,14594000.0,3226000.0,3325000.0,21145000.0
부산,11531000.0,4857000.0,2366000.0,18754000.0
서울,11914000.0,9244000.0,2322000.0,23480000.0
인천,17372000.0,9675000.0,530000.0,27577000.0
합계,55411000.0,27002000.0,8543000.0,90956000.0


[카테고리별 KPI]


,총매출,주문건수,객단가
category,,,
가전,20248000.0,398,50874.0
뷰티,9986000.0,192,52010.0
식품,22404000.0,398,56291.0
패션,38318000.0,512,74840.0



[주문별 지역 매출 점유율 — 앞 10행]


,order_id,region,amount,share_in_region(%)
0,O00001,서울,120000.0,0.511
1,O00002,서울,25000.0,0.106
2,O00003,인천,36000.0,0.131
3,O00004,경기,120000.0,0.568
4,O00005,경기,120000.0,0.568
5,O00006,서울,36000.0,0.153
6,O00007,서울,25000.0,0.106
7,O00008,부산,75000.0,0.400
8,O00009,인천,75000.0,0.272
9,O00010,부산,150000.0,0.800


In [14]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn -q

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False   # 마이너스 부호 깨짐 방지
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [15]:
# ─────────────────────────────────────────────
# 모두마켓 데이터 생성 — 이 셀 하나로 오늘 쓸 세 테이블이 모두 준비됩니다.
# 오늘의 초점은 '병합'이므로, 날짜는 분석 가능한 형태(datetime)로 깔끔하게 둡니다.
# (문자열·날짜 오염 정제는 다음 시간 D+005의 주제입니다.)
# ─────────────────────────────────────────────
np.random.seed(42)

# 1) 고객(customers) — 고객 1명이 한 행
n_customers = 300
customers = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_customers + 1)],
    "age": np.random.normal(35, 9, n_customers).round().astype(int).clip(18, 70),
    "gender": np.random.choice(["M", "F"], n_customers),
    "region": np.random.choice(["서울", "경기", "부산", "인천", "대구"], n_customers),
    "membership": np.random.choice(["basic", "premium", "vip"], n_customers, p=[0.6, 0.3, 0.1]),
})

# 2) 상품(products) — 상품 1종이 한 행
categories = ["패션", "뷰티", "식품", "가전", "도서"]
n_products = 40
products = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_products + 1)],
    "category": np.random.choice(categories, n_products),
    "price": np.random.choice([9900, 19900, 29900, 49900, 89900, 129900], n_products),
})

# 3) 주문(orders) — 주문 1건이 한 행. customer_id·product_id로 위 두 표와 연결됩니다.
n_orders = 2000
order_customer = np.random.choice(customers["customer_id"], n_orders)
order_product = np.random.choice(products["product_id"], n_orders)
price_map = products.set_index("product_id")["price"]
quantity = np.random.choice([1, 1, 1, 2, 2, 3], n_orders)
amount = price_map.loc[order_product].values * quantity
order_dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 120, n_orders), unit="D")

orders = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_orders + 1)],
    "customer_id": order_customer,
    "product_id": order_product,
    "quantity": quantity,
    "amount": amount.astype(float),
    "channel": np.random.choice(["web", "app"], n_orders, p=[0.5, 0.5]),
    "order_date": order_dates,
})

print("모두마켓 데이터 생성 완료")
print("customers:", customers.shape, "| products:", products.shape, "| orders:", orders.shape)

모두마켓 데이터 생성 완료
customers: (300, 5) | products: (40, 3) | orders: (2000, 7)


In [16]:
# ─────────────────────────────────────────────
# 종합 실습용 데이터 — 새 스냅샷 (이 셀부터 단독 실행 가능)
# ─────────────────────────────────────────────
np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})
print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

스냅샷 준비 완료 — orders: (1500, 6) | customers: (200, 3) | products: (30, 3)


In [13]:
# (1) 검증: 룩업 표의 키가 유일한가, 주문의 키가 모두 매칭되는가

print(ordr.columns)
print(cust.columns)
print(prod.columns)

print("[병합 전 검증]")
print("  customers 키 중복:", cust["customer_id"].duplicated().sum(), "건")
print("  products  키 중복:", prod["product_id"].duplicated().sum(), "건")
print("  매칭 안 되는 customer_id:", (~ordr["customer_id"].isin(cust["customer_id"])).sum(), "건")
print("  매칭 안 되는 product_id :", (~ordr["product_id"].isin(prod["product_id"])).sum(), "건")

# (2) 병합: validate로 관계 가정(m:1)을 강제. indicator로 매칭 확인. (병합된 테이블 명 df)
df = (
    ordr
    .merge(cust, on="customer_id", how="left", validate="m:1", indicator="cust_merge")
    .merge(prod, on="product_id", how="left", validate="m:1", indicator="prod_merge")
)
print("\n[병합 결과] 행 수:", len(df), "(원본 주문 수와 같으면 폭증 없음)")
display(df.head(3))
print(df["cust_merge"].value_counts())

Index(['order_id', 'customer_id', 'product_id', 'quantity', 'amount',
       'order_date'],
      dtype='str')
Index(['customer_id', 'region', 'membership'], dtype='str')
Index(['product_id', 'category', 'price'], dtype='str')
[병합 전 검증]
  customers 키 중복: 0 건
  products  키 중복: 0 건
  매칭 안 되는 customer_id: 0 건
  매칭 안 되는 product_id : 0 건

[병합 결과] 행 수: 1500 (원본 주문 수와 같으면 폭증 없음)


,order_id,customer_id,product_id,quantity,amount,order_date,region,membership,cust_merge,category,price,prod_merge
0,O00001,C0032,P005,3,120000.0,2025-04-12,서울,premium,both,패션,40000,both
1,O00002,C0049,P020,1,25000.0,2025-03-09,서울,basic,both,식품,25000,both
2,O00003,C0041,P016,3,36000.0,2025-01-27,인천,basic,both,패션,12000,both


cust_merge
both          1500
left_only        0
right_only       0
Name: count, dtype: int64
